In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Phase 2 first model logs:

python train_expA.py 
Device: cuda
Train samples: 9168  |  Val samples: 933
Computing damage class weights...
Damage class weights: [0.0, 0.12900076806545258, 1.1411386728286743, 1.0950143337249756, 1.6348462104797363]
Loaded Phase 1 weights | missing: 92 | unexpected: 0
Epoch 01 | train loc 0.0106 dmg 0.0043 acc 0.173 | val loc 0.0119 dmg 0.0091 acc 0.202 | ✓        
Epoch 02 | train loc 0.0060 dmg 0.0031 acc 0.218 | val loc 0.0111 dmg 0.0072 acc 0.238 | ✓        
Epoch 03 | train loc 0.0057 dmg 0.0027 acc 0.246 | val loc 0.0115 dmg 0.0061 acc 0.360 | ✓        
Epoch 04 | train loc 0.0054 dmg 0.0026 acc 0.274 | val loc 0.0110 dmg 0.0062 acc 0.333 | ✗        
Epoch 05 | train loc 0.0053 dmg 0.0023 acc 0.295 | val loc 0.0109 dmg 0.0055 acc 0.335 | ✓        
Epoch 06 | train loc 0.0052 dmg 0.0023 acc 0.290 | val loc 0.0108 dmg 0.0055 acc 0.291 | ✗        
Epoch 07 | train loc 0.0051 dmg 0.0021 acc 0.280 | val loc 0.0110 dmg 0.0052 acc 0.307 | ✓        
Epoch 08 | train loc 0.0049 dmg 0.0019 acc 0.318 | val loc 0.0108 dmg 0.0054 acc 0.392 | ✗        
Epoch 09 | train loc 0.0050 dmg 0.0019 acc 0.317 | val loc 0.0107 dmg 0.0050 acc 0.278 | ✓        
Epoch 10 | train loc 0.0048 dmg 0.0018 acc 0.337 | val loc 0.0109 dmg 0.0068 acc 0.336 | ✗        
Epoch 11 | train loc 0.0047 dmg 0.0016 acc 0.339 | val loc 0.0106 dmg 0.0053 acc 0.389 | ✗        
Epoch 12 | train loc 0.0046 dmg 0.0015 acc 0.389 | val loc 0.0109 dmg 0.0047 acc 0.336 | ✓        
Epoch 13 | train loc 0.0046 dmg 0.0014 acc 0.390 | val loc 0.0115 dmg 0.0056 acc 0.313 | ✗        
Epoch 14 | train loc 0.0045 dmg 0.0013 acc 0.405 | val loc 0.0109 dmg 0.0062 acc 0.409 | ✗        
Epoch 15 | train loc 0.0045 dmg 0.0013 acc 0.376 | val loc 0.0126 dmg 0.0085 acc 0.235 | ✗        
Epoch 16 | train loc 0.0044 dmg 0.0012 acc 0.409 | val loc 0.0110 dmg 0.0049 acc 0.409 | ✗        
Epoch 17 | train loc 0.0043 dmg 0.0011 acc 0.438 | val loc 0.0112 dmg 0.0064 acc 0.461 | ✗        
Epoch 18/40 Train:  40%|█▌  | 913/2292 [12:52<19:26,  1.18it/s, acc=0.452, dmg=0.0009, loc=0.0042]

# dataset_expA.py

In [1]:
import os
import glob
import random

import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


class XBDJointDataset(Dataset):
    def __init__(self, splits, base_dir="xbd", augment=False):
        self.augment = augment
        self.samples = []  # (pre_img, post_img, pre_mask, post_mask)

        for split in splits:
            img_dir  = os.path.join(base_dir, split, "images")
            mask_dir = os.path.join(base_dir, split, "masks")
            if not os.path.isdir(img_dir):
                continue

            for pre_img_path in glob.glob(os.path.join(img_dir, "*_pre_disaster.png")):
                stem          = os.path.basename(pre_img_path).replace("_pre_disaster.png", "")
                post_img_path  = os.path.join(img_dir,  f"{stem}_post_disaster.png")
                pre_mask_path  = os.path.join(mask_dir, f"{stem}_pre_disaster.png")
                post_mask_path = os.path.join(mask_dir, f"{stem}_post_disaster.png")

                if not all(os.path.exists(p) for p in [post_img_path, pre_mask_path, post_mask_path]):
                    continue

                self.samples.append((pre_img_path, post_img_path, pre_mask_path, post_mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pre_img_path, post_img_path, pre_mask_path, post_mask_path = self.samples[idx]

        pre_img  = Image.open(pre_img_path).convert("RGB")
        post_img = Image.open(post_img_path).convert("RGB")
        pre_mask = Image.open(pre_mask_path)   # 0 or 255
        post_mask = Image.open(post_mask_path)  # 0-4

        if self.augment:
            if random.random() > 0.5:
                pre_img, post_img = TF.hflip(pre_img), TF.hflip(post_img)
                pre_mask, post_mask = TF.hflip(pre_mask), TF.hflip(post_mask)
            if random.random() > 0.5:
                pre_img, post_img = TF.vflip(pre_img), TF.vflip(post_img)
                pre_mask, post_mask = TF.vflip(pre_mask), TF.vflip(post_mask)

        pre_t  = TF.normalize(TF.to_tensor(pre_img),  MEAN, STD)
        post_t = TF.normalize(TF.to_tensor(post_img), MEAN, STD)

        # pre mask: 255 -> 1 binary
        loc_mask = torch.from_numpy(np.array(pre_mask)).float() / 255.0  # (H, W) in [0,1]

        # post mask: values 0-4 directly as class indices
        dmg_mask = torch.from_numpy(np.array(post_mask)).long()           # (H, W) in {0,1,2,3,4}

        return pre_t, post_t, loc_mask.unsqueeze(0), dmg_mask


# model_expA.py

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels=256):
        super().__init__()
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=True),
            nn.GroupNorm(32, out_channels), nn.ReLU()
        )
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False),
                          nn.BatchNorm2d(out_channels), nn.ReLU()),
            *[nn.Sequential(nn.Conv2d(in_channels, out_channels, 3,
                                      padding=r, dilation=r, bias=False),
                            nn.BatchNorm2d(out_channels), nn.ReLU())
              for r in (6, 12, 18)],
        ])
        self.proj = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Dropout(0.5)
        )

    def forward(self, x):
        size = x.shape[2:]
        pool = F.interpolate(self.global_pool(x), size, mode="bilinear", align_corners=False)
        feats = [pool] + [b(x) for b in self.branches]
        return self.proj(torch.cat(feats, dim=1))


class DecoderHead(nn.Module):
    """Shared DeepLabV3+ style decoder, parameterized by num_classes."""
    def __init__(self, num_classes, aspp_in=1024, low_in=256):
        super().__init__()
        self.aspp    = ASPP(aspp_in * 2)   # *2 because pre+post concat
        self.low_proj = nn.Sequential(
            nn.Conv2d(low_in * 2, 48, 1, bias=False),
            nn.BatchNorm2d(48), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, num_classes, 1)
        )

    def forward(self, low, high):
        aspp = self.aspp(high)
        aspp = F.interpolate(aspp, low.shape[2:], mode="bilinear", align_corners=False)
        x    = torch.cat([aspp, self.low_proj(low)], dim=1)
        x    = self.decoder(x)
        return F.interpolate(x, scale_factor=4, mode="bilinear", align_corners=False)


class JointDamageNet(nn.Module):
    def __init__(self, loc_classes=1, dmg_classes=5):
        super().__init__()
        backbone = resnet50(weights=None)

        self.low_level = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1
        )  # -> (B, 256, H/4, W/4)

        layer3 = backbone.layer3
        for m in layer3.modules():
            if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
                m.stride = (1, 1)
            if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3):
                m.dilation = (2, 2)
                m.padding  = (2, 2)

        self.high_level = nn.Sequential(backbone.layer2, layer3)
        # -> (B, 1024, H/16, W/16) — dilated so stays at stride 16

        self.loc_head = DecoderHead(num_classes=loc_classes)
        self.dmg_head = DecoderHead(num_classes=dmg_classes)

    def _encode(self, x):
        low  = self.low_level(x)
        high = self.high_level(low)
        return low, high

    def forward(self, pre, post):
        pre_low,  pre_high  = self._encode(pre)
        post_low, post_high = self._encode(post)

        # Fuse by concatenating along channel dim
        low  = torch.cat([pre_low,  post_low],  dim=1)  # (B, 512,  H/4,  W/4)
        high = torch.cat([pre_high, post_high], dim=1)  # (B, 2048, H/16, W/16)

        loc_out = self.loc_head(low, high)  # (B, 1, H, W)
        dmg_out = self.dmg_head(low, high)  # (B, 5, H, W)

        return loc_out, dmg_out

    def load_phase1_weights(self, ckpt_path, device):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        state = ckpt["model_state_dict"]

        # Map Phase 1 keys -> this model's encoder keys (low_level, high_level)
        new_state = {}
        for k, v in state.items():
            if k.startswith("low_level.") or k.startswith("high_level."):
                new_state[k] = v

        missing, unexpected = self.load_state_dict(new_state, strict=False)
        print(f"Loaded Phase 1 weights | missing: {len(missing)} | unexpected: {len(unexpected)}")


# train_expA.py

In [7]:
import os
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset_expA import XBDJointDataset
from model_expA import JointDamageNet

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR        = "xbd"
TRAIN_SPLITS    = ["tier1", "tier3"]
VAL_SPLITS      = ["hold"]
BATCH_SIZE      = 4
NUM_WORKERS     = 4
LR              = 1e-4
EPOCHS          = 40
FOCAL_GAMMA     = 2.0
LOC_LOSS_WEIGHT = 1.0   # λ for localization loss
DMG_LOSS_WEIGHT = 1.0   # λ for damage loss
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH       = "best_expA_model.pth"
PHASE1_CKPT     = "best_model.pth"
# ──────────────────────────────────────────────────────────────────────────────


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=0):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight,
                               ignore_index=self.ignore_index, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        # ignore_index positions have ce=0, safe to mean over all
        return loss.mean()


class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt  = torch.exp(-bce)
        return (((1 - pt) ** self.gamma) * bce).mean()


def compute_dmg_weights(dataset):
    counts = collections.Counter()
    for _, _, _, post_mask_path in dataset.samples:
        from PIL import Image
        import numpy as np
        arr = np.array(Image.open(post_mask_path)).flatten()
        for v in range(1, 5):   # skip background
            counts[v] += (arr == v).sum()
    total = sum(counts.values())
    weights = torch.tensor(
        [total / (4 * max(counts[c], 1)) for c in range(1, 5)], dtype=torch.float32
    )
    weights = weights / weights.sum() * 4
    return torch.cat([torch.zeros(1), weights])  # prepend 0 for background


def run_epoch(model, loader, loc_criterion, dmg_criterion, optimizer, train, epoch, num_epochs):
    model.train(train)
    total_loc, total_dmg, total_dmg_acc, n = 0.0, 0.0, 0.0, 0
    phase = "Train" if train else "Val"
    pbar  = tqdm(loader, desc=f"Epoch {epoch:02d}/{num_epochs} {phase}", leave=False)

    with torch.set_grad_enabled(train):
        for pre, post, loc_mask, dmg_mask in pbar:
            pre      = pre.to(DEVICE)
            post     = post.to(DEVICE)
            loc_mask = loc_mask.to(DEVICE)
            dmg_mask = dmg_mask.to(DEVICE)

            loc_out, dmg_out = model(pre, post)

            loc_loss = loc_criterion(loc_out, loc_mask)
            dmg_loss = dmg_criterion(dmg_out, dmg_mask)
            loss     = LOC_LOSS_WEIGHT * loc_loss + DMG_LOSS_WEIGHT * dmg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = pre.size(0)
            total_loc     += loc_loss.item() * bs
            total_dmg     += dmg_loss.item() * bs
            # accuracy only on building pixels (dmg_mask > 0)
            mask           = dmg_mask > 0
            if mask.any():
                total_dmg_acc += (dmg_out.argmax(1)[mask] == dmg_mask[mask]).float().mean().item() * bs
            n             += bs

            pbar.set_postfix(loc=f"{total_loc/n:.4f}", dmg=f"{total_dmg/n:.4f}",
                             acc=f"{total_dmg_acc/n:.3f}")

    return total_loc / n, total_dmg / n, total_dmg_acc / n


def main():
    print(f"Device: {DEVICE}")

    train_ds = XBDJointDataset(TRAIN_SPLITS, BASE_DIR, augment=True)
    val_ds   = XBDJointDataset(VAL_SPLITS,   BASE_DIR, augment=False)
    print(f"Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}")

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print("Computing damage class weights...")
    dmg_weights   = compute_dmg_weights(train_ds).to(DEVICE)
    print("Damage class weights:", dmg_weights.tolist())

    model = JointDamageNet(loc_classes=1, dmg_classes=5).to(DEVICE)
    model.load_phase1_weights(PHASE1_CKPT, DEVICE)

    loc_criterion = BinaryFocalLoss(gamma=FOCAL_GAMMA)
    dmg_criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=dmg_weights, ignore_index=0)
    optimizer     = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS,
                                                              last_epoch=start_epoch - 2)

    best_val_dmg = float("inf")
    start_epoch   = 1

    if os.path.exists(SAVE_PATH):
        print(f"Resuming from checkpoint: {SAVE_PATH}")
        ckpt          = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_val_dmg  = ckpt["val_dmg_loss"]
        start_epoch   = ckpt["epoch"] + 1
        print(f"Resumed at epoch {start_epoch}, best val dmg loss: {best_val_dmg:.4f}")

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loc, tr_dmg, tr_acc = run_epoch(model, train_loader, loc_criterion, dmg_criterion,
                                           optimizer, train=True,  epoch=epoch, num_epochs=EPOCHS)
        vl_loc, vl_dmg, vl_acc = run_epoch(model, val_loader,   loc_criterion, dmg_criterion,
                                           optimizer, train=False, epoch=epoch, num_epochs=EPOCHS)
        scheduler.step()

        improved = vl_dmg < best_val_dmg
        if improved:
            best_val_dmg = vl_dmg
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dmg_loss": vl_dmg,
            }, SAVE_PATH)

        status = "✓" if improved else "✗"
        print(f"Epoch {epoch:02d} | "
              f"train loc {tr_loc:.4f} dmg {tr_dmg:.4f} acc {tr_acc:.3f} | "
              f"val loc {vl_loc:.4f} dmg {vl_dmg:.4f} acc {vl_acc:.3f} | {status}")

    print(f"\nBest val dmg loss: {best_val_dmg:.4f} — saved to {SAVE_PATH}")


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'dataset_expA'

# model_expB.py

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels=256):
        super().__init__()
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=True),
            nn.GroupNorm(32, out_channels), nn.ReLU()
        )
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False),
                          nn.BatchNorm2d(out_channels), nn.ReLU()),
            *[nn.Sequential(nn.Conv2d(in_channels, out_channels, 3,
                                      padding=r, dilation=r, bias=False),
                            nn.BatchNorm2d(out_channels), nn.ReLU())
              for r in (6, 12, 18)],
        ])
        self.proj = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Dropout(0.5)
        )

    def forward(self, x):
        size = x.shape[2:]
        pool = F.interpolate(self.global_pool(x), size, mode="bilinear", align_corners=False)
        feats = [pool] + [b(x) for b in self.branches]
        return self.proj(torch.cat(feats, dim=1))


class DecoderHead(nn.Module):
    def __init__(self, num_classes, aspp_in=1024, low_in=256, dropout=0.3):
        super().__init__()
        self.aspp     = ASPP(aspp_in * 2)
        self.low_proj = nn.Sequential(
            nn.Conv2d(low_in * 2, 48, 1, bias=False),
            nn.BatchNorm2d(48), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(256, num_classes, 1)
        )

    def forward(self, low, high):
        aspp = self.aspp(high)
        aspp = F.interpolate(aspp, low.shape[2:], mode="bilinear", align_corners=False)
        x    = torch.cat([aspp, self.low_proj(low)], dim=1)
        x    = self.decoder(x)
        return F.interpolate(x, scale_factor=4, mode="bilinear", align_corners=False)


class JointDamageNet(nn.Module):
    def __init__(self, loc_classes=1, dmg_classes=5, dropout=0.3):
        super().__init__()
        backbone = resnet50(weights=None)

        self.low_level = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1
        )  # -> (B, 256, H/4, W/4)

        layer3 = backbone.layer3
        for m in layer3.modules():
            if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
                m.stride = (1, 1)
            if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3):
                m.dilation = (2, 2)
                m.padding  = (2, 2)

        self.high_level = nn.Sequential(backbone.layer2, layer3)
        # -> (B, 1024, H/16, W/16)

        self.loc_head = DecoderHead(num_classes=loc_classes, dropout=dropout)
        self.dmg_head = DecoderHead(num_classes=dmg_classes, dropout=dropout)

    def _encode(self, x):
        low  = self.low_level(x)
        high = self.high_level(low)
        return low, high

    def forward(self, pre, post):
        pre_low,  pre_high  = self._encode(pre)
        post_low, post_high = self._encode(post)

        low  = torch.cat([pre_low,  post_low],  dim=1)
        high = torch.cat([pre_high, post_high], dim=1)

        loc_out = self.loc_head(low, high)  # (B, 1, H, W)
        dmg_out = self.dmg_head(low, high)  # (B, 5, H, W)

        return loc_out, dmg_out

    def load_phase1_weights(self, ckpt_path, device):
        ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
        state = ckpt["model_state_dict"]

        new_state = {k: v for k, v in state.items()
                     if k.startswith("low_level.") or k.startswith("high_level.")}

        missing, unexpected = self.load_state_dict(new_state, strict=False)
        print(f"Loaded Phase 1 weights | missing: {len(missing)} | unexpected: {len(unexpected)}")


# train_expB.py

In [1]:
import os
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset_expB import XBDJointDataset
from model_expB import JointDamageNet

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR        = "xbd"
TRAIN_SPLITS    = ["tier1", "tier3"]
VAL_SPLITS      = ["hold"]
BATCH_SIZE      = 8          # increased from 4
NUM_WORKERS     = 4
LR              = 1e-4
ENCODER_LR_MULT = 0.3        # encoder gets LR * 0.3
EPOCHS          = 40
WARMUP_EPOCHS   = 3
FOCAL_GAMMA     = 2.0
LOC_LOSS_WEIGHT = 1.0
DMG_LOSS_WEIGHT = 1.0
DROPOUT         = 0.3
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH       = "best_expB_model.pth"
PHASE1_CKPT     = "best_model.pth"
# ──────────────────────────────────────────────────────────────────────────────


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=0):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight,
                               ignore_index=self.ignore_index, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt  = torch.exp(-bce)
        return (((1 - pt) ** self.gamma) * bce).mean()


def compute_dmg_weights(dataset):
    counts = collections.Counter()
    for _, _, _, post_mask_path, _, _ in dataset.tiles:
        from PIL import Image
        import numpy as np
        arr = np.array(Image.open(post_mask_path)).flatten()
        for v in range(1, 5):
            counts[v] += (arr == v).sum()
    total = sum(counts.values())
    weights = torch.tensor(
        [total / (4 * max(counts[c], 1)) for c in range(1, 5)], dtype=torch.float32
    )
    weights = weights / weights.sum() * 4
    return torch.cat([torch.zeros(1), weights])


def run_epoch(model, loader, loc_criterion, dmg_criterion, optimizer, scaler, train, epoch, num_epochs):
    model.train(train)
    total_loc, total_dmg, total_dmg_correct, total_dmg_pixels, n = 0.0, 0.0, 0, 0, 0
    phase = "Train" if train else "Val"
    pbar  = tqdm(loader, desc=f"Epoch {epoch:02d}/{num_epochs} {phase}", leave=False)

    with torch.set_grad_enabled(train):
        for pre, post, loc_mask, dmg_mask in pbar:
            pre      = pre.to(DEVICE)
            post     = post.to(DEVICE)
            loc_mask = loc_mask.to(DEVICE)
            dmg_mask = dmg_mask.to(DEVICE)

            with autocast():
                loc_out, dmg_out = model(pre, post)
                loc_loss = loc_criterion(loc_out, loc_mask)
                dmg_loss = dmg_criterion(dmg_out, dmg_mask)
                loss     = LOC_LOSS_WEIGHT * loc_loss + DMG_LOSS_WEIGHT * dmg_loss

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            bs   = pre.size(0)
            mask = dmg_mask > 0
            total_loc          += loc_loss.item() * bs
            total_dmg          += dmg_loss.item() * bs
            total_dmg_correct  += (dmg_out.argmax(1)[mask] == dmg_mask[mask]).sum().item()
            total_dmg_pixels   += mask.sum().item()
            n += bs

            pbar.set_postfix(loc=f"{total_loc/n:.4f}", dmg=f"{total_dmg/n:.4f}",
                             acc=f"{total_dmg_correct/max(total_dmg_pixels,1):.3f}")

    return total_loc / n, total_dmg / n, total_dmg_correct / max(total_dmg_pixels, 1)


def main():
    print(f"Device: {DEVICE}")

    train_ds = XBDJointDataset(TRAIN_SPLITS, BASE_DIR, augment=True)
    val_ds   = XBDJointDataset(VAL_SPLITS,   BASE_DIR, augment=False)
    print(f"Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}")

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print("Computing damage class weights...")
    dmg_weights = compute_dmg_weights(train_ds).to(DEVICE)
    print("Damage class weights:", dmg_weights.tolist())

    model = JointDamageNet(loc_classes=1, dmg_classes=5, dropout=DROPOUT).to(DEVICE)
    model.load_phase1_weights(PHASE1_CKPT, DEVICE)

    loc_criterion = BinaryFocalLoss(gamma=FOCAL_GAMMA)
    dmg_criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=dmg_weights, ignore_index=0)

    # Differential LRs — encoder gets 3x lower LR than decoder heads
    optimizer = torch.optim.AdamW([
        {"params": model.low_level.parameters(),  "lr": LR * ENCODER_LR_MULT},
        {"params": model.high_level.parameters(), "lr": LR * ENCODER_LR_MULT},
        {"params": model.loc_head.parameters(),   "lr": LR},
        {"params": model.dmg_head.parameters(),   "lr": LR},
    ], weight_decay=1e-4)

    scaler = GradScaler()

    best_val_acc = 0.0
    start_epoch  = 1

    if os.path.exists(SAVE_PATH):
        print(f"Resuming from checkpoint: {SAVE_PATH}")
        ckpt         = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_val_acc = ckpt.get("val_acc", 0.0)
        start_epoch  = ckpt["epoch"] + 1
        print(f"Resumed at epoch {start_epoch}, best val acc: {best_val_acc:.4f}")

    warmup    = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                                                   total_iters=WARMUP_EPOCHS)
    cosine    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine],
                                                       milestones=[WARMUP_EPOCHS])

    if os.path.exists(SAVE_PATH):
        ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        if "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loc, tr_dmg, tr_acc = run_epoch(model, train_loader, loc_criterion, dmg_criterion,
                                           optimizer, scaler, train=True,  epoch=epoch, num_epochs=EPOCHS)
        vl_loc, vl_dmg, vl_acc = run_epoch(model, val_loader,   loc_criterion, dmg_criterion,
                                           optimizer, scaler, train=False, epoch=epoch, num_epochs=EPOCHS)
        scheduler.step()

        improved = vl_acc > best_val_acc
        if improved:
            best_val_acc = vl_acc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "val_acc": vl_acc,
            }, SAVE_PATH)

        status = "✓" if improved else "✗"
        print(f"Epoch {epoch:02d} | "
              f"train loc {tr_loc:.4f} dmg {tr_dmg:.4f} acc {tr_acc:.3f} | "
              f"val loc {vl_loc:.4f} dmg {vl_dmg:.4f} acc {vl_acc:.3f} | {status}")

    print(f"\nBest val acc: {best_val_acc:.4f} — saved to {SAVE_PATH}")


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'dataset_expB'

# expA ruuning 2 logs:

Epoch 16 | train loc 0.0044 dmg 0.0011 acc 0.426 | val loc 0.0106 dmg 0.0059 acc 0.439 | ✓        
Epoch 17 | train loc 0.0043 dmg 0.0010 acc 0.455 | val loc 0.0108 dmg 0.0056 acc 0.405 | ✗        
Epoch 18 | train loc 0.0042 dmg 0.0009 acc 0.462 | val loc 0.0110 dmg 0.0050 acc 0.394 | ✗        
Epoch 19 | train loc 0.0042 dmg 0.0010 acc 0.450 | val loc 0.0111 dmg 0.0059 acc 0.447 | ✓        
Epoch 20 | train loc 0.0041 dmg 0.0008 acc 0.483 | val loc 0.0114 dmg 0.0067 acc 0.463 | ✓        
Epoch 21 | train loc 0.0040 dmg 0.0007 acc 0.514 | val loc 0.0110 dmg 0.0073 acc 0.497 | ✓        
Epoch 22 | train loc 0.0039 dmg 0.0007 acc 0.533 | val loc 0.0116 dmg 0.0072 acc 0.527 | ✓        
Epoch 23 | train loc 0.0039 dmg 0.0006 acc 0.540 | val loc 0.0115 dmg 0.0072 acc 0.473 | ✗        
Epoch 24 | train loc 0.0038 dmg 0.0006 acc 0.548 | val loc 0.0118 dmg 0.0080 acc 0.578 | ✓        
Epoch 25 | train loc 0.0037 dmg 0.0005 acc 0.560 | val loc 0.0114 dmg 0.0071 acc 0.476 | ✗        
Epoch 26 | train loc 0.0037 dmg 0.0004 acc 0.588 | val loc 0.0118 dmg 0.0093 acc 0.604 | ✓        
Epoch 27 | train loc 0.0036 dmg 0.0004 acc 0.611 | val loc 0.0120 dmg 0.0087 acc 0.566 | ✗        
Epoch 28 | train loc 0.0036 dmg 0.0003 acc 0.625 | val loc 0.0124 dmg 0.0102 acc 0.598 | ✗        
Epoch 29 | train loc 0.0035 dmg 0.0003 acc 0.654 | val loc 0.0127 dmg 0.0098 acc 0.605 | ✓        
Epoch 30 | train loc 0.0035 dmg 0.0003 acc 0.643 | val loc 0.0127 dmg 0.0101 acc 0.637 | ✓        
Epoch 31 | train loc 0.0034 dmg 0.0003 acc 0.664 | val loc 0.0128 dmg 0.0111 acc 0.654 | ✓        
Epoch 32 | train loc 0.0034 dmg 0.0002 acc 0.681 | val loc 0.0131 dmg 0.0121 acc 0.667 | ✓        
Epoch 33 | train loc 0.0033 dmg 0.0002 acc 0.685 | val loc 0.0136 dmg 0.0115 acc 0.641 | ✗        
Epoch 34 | train loc 0.0033 dmg 0.0002 acc 0.684 | val loc 0.0134 dmg 0.0116 acc 0.646 | ✗        
Epoch 35 | train loc 0.0033 dmg 0.0002 acc 0.700 | val loc 0.0133 dmg 0.0120 acc 0.663 | ✗        
Epoch 36 | train loc 0.0033 dmg 0.0002 acc 0.690 | val loc 0.0133 dmg 0.0128 acc 0.673 | ✓        
Epoch 37 | train loc 0.0033 dmg 0.0002 acc 0.705 | val loc 0.0135 dmg 0.0135 acc 0.670 | ✗        
Epoch 38 | train loc 0.0032 dmg 0.0002 acc 0.705 | val loc 0.0136 dmg 0.0124 acc 0.656 | ✗        
Epoch 39 | train loc 0.0033 dmg 0.0002 acc 0.706 | val loc 0.0135 dmg 0.0130 acc 0.659 | ✗        
Epoch 40 | train loc 0.0032 dmg 0.0002 acc 0.705 | val loc 0.0138 dmg 0.0130 acc 0.665 | ✗        

Best val acc: 0.6733 — saved to best_expA_model.pth

# model_expB.py


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels=256):
        super().__init__()
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=True),
            nn.GroupNorm(32, out_channels), nn.ReLU()
        )
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False),
                          nn.BatchNorm2d(out_channels), nn.ReLU()),
            *[nn.Sequential(nn.Conv2d(in_channels, out_channels, 3,
                                      padding=r, dilation=r, bias=False),
                            nn.BatchNorm2d(out_channels), nn.ReLU())
              for r in (6, 12, 18)],
        ])
        self.proj = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Dropout(0.5)
        )

    def forward(self, x):
        size = x.shape[2:]
        pool = F.interpolate(self.global_pool(x), size, mode="bilinear", align_corners=False)
        feats = [pool] + [b(x) for b in self.branches]
        return self.proj(torch.cat(feats, dim=1))


class DecoderHead(nn.Module):
    def __init__(self, num_classes, aspp_in=1024, low_in=256, dropout=0.3):
        super().__init__()
        self.aspp     = ASPP(aspp_in * 2)
        self.low_proj = nn.Sequential(
            nn.Conv2d(low_in * 2, 48, 1, bias=False),
            nn.BatchNorm2d(48), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(256, num_classes, 1)
        )

    def forward(self, low, high):
        aspp = self.aspp(high)
        aspp = F.interpolate(aspp, low.shape[2:], mode="bilinear", align_corners=False)
        x    = torch.cat([aspp, self.low_proj(low)], dim=1)
        x    = self.decoder(x)
        return F.interpolate(x, scale_factor=4, mode="bilinear", align_corners=False)


class JointDamageNet(nn.Module):
    def __init__(self, loc_classes=1, dmg_classes=5, dropout=0.3):
        super().__init__()
        backbone = resnet50(weights=None)

        self.low_level = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1
        )  # -> (B, 256, H/4, W/4)

        layer3 = backbone.layer3
        for m in layer3.modules():
            if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
                m.stride = (1, 1)
            if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3):
                m.dilation = (2, 2)
                m.padding  = (2, 2)

        self.high_level = nn.Sequential(backbone.layer2, layer3)
        # -> (B, 1024, H/16, W/16)

        self.loc_head = DecoderHead(num_classes=loc_classes, dropout=dropout)
        self.dmg_head = DecoderHead(num_classes=dmg_classes, dropout=dropout)

    def _encode(self, x):
        low  = self.low_level(x)
        high = self.high_level(low)
        return low, high

    def forward(self, pre, post):
        pre_low,  pre_high  = self._encode(pre)
        post_low, post_high = self._encode(post)

        low  = torch.cat([pre_low,  post_low],  dim=1)
        high = torch.cat([pre_high, post_high], dim=1)

        loc_out = self.loc_head(low, high)  # (B, 1, H, W)
        dmg_out = self.dmg_head(low, high)  # (B, 5, H, W)

        return loc_out, dmg_out

    def load_phase1_weights(self, ckpt_path, device):
        ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
        state = ckpt["model_state_dict"]

        new_state = {k: v for k, v in state.items()
                     if k.startswith("low_level.") or k.startswith("high_level.")}

        missing, unexpected = self.load_state_dict(new_state, strict=False)
        print(f"Loaded Phase 1 weights | missing: {len(missing)} | unexpected: {len(unexpected)}")


In [2]:
import os
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset_expA import XBDJointDataset
from model_expB import JointDamageNet

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR        = "xbd"
TRAIN_SPLITS    = ["tier1", "tier3"]
VAL_SPLITS      = ["hold"]
BATCH_SIZE      = 8          # increased from 4
NUM_WORKERS     = 4
LR              = 1e-4
ENCODER_LR_MULT = 0.1        # encoder gets LR * 0.1
EPOCHS          = 40
FOCAL_GAMMA     = 2.0
LOC_LOSS_WEIGHT = 1.0
DMG_LOSS_WEIGHT = 1.0
DROPOUT         = 0.3
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH       = "best_expB_model.pth"
PHASE1_CKPT     = "best_model.pth"
# ──────────────────────────────────────────────────────────────────────────────


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=0):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight,
                               ignore_index=self.ignore_index, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt  = torch.exp(-bce)
        return (((1 - pt) ** self.gamma) * bce).mean()


def compute_dmg_weights(dataset):
    counts = collections.Counter()
    for _, _, _, post_mask_path in dataset.samples:
        from PIL import Image
        import numpy as np
        arr = np.array(Image.open(post_mask_path)).flatten()
        for v in range(1, 5):
            counts[v] += (arr == v).sum()
    total = sum(counts.values())
    weights = torch.tensor(
        [total / (4 * max(counts[c], 1)) for c in range(1, 5)], dtype=torch.float32
    )
    weights = weights / weights.sum() * 4
    return torch.cat([torch.zeros(1), weights])


def run_epoch(model, loader, loc_criterion, dmg_criterion, optimizer, scaler, train, epoch, num_epochs):
    model.train(train)
    total_loc, total_dmg, total_dmg_acc, n = 0.0, 0.0, 0.0, 0
    phase = "Train" if train else "Val"
    pbar  = tqdm(loader, desc=f"Epoch {epoch:02d}/{num_epochs} {phase}", leave=False)

    with torch.set_grad_enabled(train):
        for pre, post, loc_mask, dmg_mask in pbar:
            pre      = pre.to(DEVICE)
            post     = post.to(DEVICE)
            loc_mask = loc_mask.to(DEVICE)
            dmg_mask = dmg_mask.to(DEVICE)

            with autocast():
                loc_out, dmg_out = model(pre, post)
                loc_loss = loc_criterion(loc_out, loc_mask)
                dmg_loss = dmg_criterion(dmg_out, dmg_mask)
                loss     = LOC_LOSS_WEIGHT * loc_loss + DMG_LOSS_WEIGHT * dmg_loss

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            bs = pre.size(0)
            total_loc += loc_loss.item() * bs
            total_dmg += dmg_loss.item() * bs
            mask = dmg_mask > 0
            if mask.any():
                total_dmg_acc += (dmg_out.argmax(1)[mask] == dmg_mask[mask]).float().mean().item() * bs
            n += bs

            pbar.set_postfix(loc=f"{total_loc/n:.4f}", dmg=f"{total_dmg/n:.4f}",
                             acc=f"{total_dmg_acc/n:.3f}")

    return total_loc / n, total_dmg / n, total_dmg_acc / n


def main():
    print(f"Device: {DEVICE}")

    train_ds = XBDJointDataset(TRAIN_SPLITS, BASE_DIR, augment=True)
    val_ds   = XBDJointDataset(VAL_SPLITS,   BASE_DIR, augment=False)
    print(f"Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}")

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print("Computing damage class weights...")
    dmg_weights = compute_dmg_weights(train_ds).to(DEVICE)
    print("Damage class weights:", dmg_weights.tolist())

    model = JointDamageNet(loc_classes=1, dmg_classes=5, dropout=DROPOUT).to(DEVICE)
    model.load_phase1_weights(PHASE1_CKPT, DEVICE)

    loc_criterion = BinaryFocalLoss(gamma=FOCAL_GAMMA)
    dmg_criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=dmg_weights, ignore_index=0)

    # Differential LRs — encoder gets 10x lower LR than decoder heads
    optimizer = torch.optim.AdamW([
        {"params": model.low_level.parameters(),  "lr": LR * ENCODER_LR_MULT},
        {"params": model.high_level.parameters(), "lr": LR * ENCODER_LR_MULT},
        {"params": model.loc_head.parameters(),   "lr": LR},
        {"params": model.dmg_head.parameters(),   "lr": LR},
    ], weight_decay=1e-4)

    scaler = GradScaler()

    best_val_acc = 0.0
    start_epoch  = 1

    if os.path.exists(SAVE_PATH):
        print(f"Resuming from checkpoint: {SAVE_PATH}")
        ckpt         = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_val_acc = ckpt.get("val_acc", 0.0)
        start_epoch  = ckpt["epoch"] + 1
        print(f"Resumed at epoch {start_epoch}, best val acc: {best_val_acc:.4f}")

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS,
                                                            last_epoch=start_epoch - 2)

    if os.path.exists(SAVE_PATH):
        ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        if "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loc, tr_dmg, tr_acc = run_epoch(model, train_loader, loc_criterion, dmg_criterion,
                                           optimizer, scaler, train=True,  epoch=epoch, num_epochs=EPOCHS)
        vl_loc, vl_dmg, vl_acc = run_epoch(model, val_loader,   loc_criterion, dmg_criterion,
                                           optimizer, scaler, train=False, epoch=epoch, num_epochs=EPOCHS)
        scheduler.step()

        improved = vl_acc > best_val_acc
        if improved:
            best_val_acc = vl_acc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "val_acc": vl_acc,
            }, SAVE_PATH)

        status = "✓" if improved else "✗"
        print(f"Epoch {epoch:02d} | "
              f"train loc {tr_loc:.4f} dmg {tr_dmg:.4f} acc {tr_acc:.3f} | "
              f"val loc {vl_loc:.4f} dmg {vl_dmg:.4f} acc {vl_acc:.3f} | {status}")

    print(f"\nBest val acc: {best_val_acc:.4f} — saved to {SAVE_PATH}")


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'dataset_expA'

# dataset_expB.py

In [4]:
import os
import glob
import random

import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

MEAN      = [0.485, 0.456, 0.406]
STD       = [0.229, 0.224, 0.225]
TILE_SIZE = 512


class XBDJointDataset(Dataset):
    def __init__(self, splits, base_dir="xbd", augment=False, min_building_ratio=0.01):
        self.augment = augment
        self.tiles   = []  # (pre_img, post_img, pre_mask, post_mask, row, col)

        for split in splits:
            img_dir  = os.path.join(base_dir, split, "images")
            mask_dir = os.path.join(base_dir, split, "masks")
            if not os.path.isdir(img_dir):
                continue

            for pre_img_path in glob.glob(os.path.join(img_dir, "*_pre_disaster.png")):
                stem           = os.path.basename(pre_img_path).replace("_pre_disaster.png", "")
                post_img_path  = os.path.join(img_dir,  f"{stem}_post_disaster.png")
                pre_mask_path  = os.path.join(mask_dir, f"{stem}_pre_disaster.png")
                post_mask_path = os.path.join(mask_dir, f"{stem}_post_disaster.png")

                if not all(os.path.exists(p) for p in [post_img_path, pre_mask_path, post_mask_path]):
                    continue

                # Use pre_mask to filter tiles by building density
                pre_mask = np.array(Image.open(pre_mask_path))
                h, w     = pre_mask.shape
                for r in range(0, h, TILE_SIZE):
                    for c in range(0, w, TILE_SIZE):
                        tile = pre_mask[r:r + TILE_SIZE, c:c + TILE_SIZE]
                        if tile.shape != (TILE_SIZE, TILE_SIZE):
                            continue
                        ratio = (tile > 0).sum() / (TILE_SIZE * TILE_SIZE)
                        if ratio >= min_building_ratio:
                            self.tiles.append((pre_img_path, post_img_path,
                                               pre_mask_path, post_mask_path, r, c))

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        pre_img_path, post_img_path, pre_mask_path, post_mask_path, r, c = self.tiles[idx]

        def crop(path, mode="RGB"):
            arr = np.array(Image.open(path).convert(mode) if mode else Image.open(path))
            return arr[r:r + TILE_SIZE, c:c + TILE_SIZE]

        pre_img   = Image.fromarray(crop(pre_img_path))
        post_img  = Image.fromarray(crop(post_img_path))
        pre_mask  = Image.fromarray(crop(pre_mask_path,  mode=None))
        post_mask = Image.fromarray(crop(post_mask_path, mode=None))

        if self.augment:
            if random.random() > 0.5:
                pre_img, post_img = TF.hflip(pre_img), TF.hflip(post_img)
                pre_mask, post_mask = TF.hflip(pre_mask), TF.hflip(post_mask)
            if random.random() > 0.5:
                pre_img, post_img = TF.vflip(pre_img), TF.vflip(post_img)
                pre_mask, post_mask = TF.vflip(pre_mask), TF.vflip(post_mask)
            k = random.randint(0, 3)
            if k:
                pre_img,  post_img  = TF.rotate(pre_img,  90*k), TF.rotate(post_img,  90*k)
                pre_mask, post_mask = TF.rotate(pre_mask, 90*k), TF.rotate(post_mask, 90*k)
            if random.random() > 0.5:
                pre_img  = TF.adjust_brightness(TF.adjust_contrast(pre_img,  random.uniform(0.8, 1.2)), random.uniform(0.8, 1.2))
                post_img = TF.adjust_brightness(TF.adjust_contrast(post_img, random.uniform(0.8, 1.2)), random.uniform(0.8, 1.2))

        pre_t  = TF.normalize(TF.to_tensor(pre_img),  MEAN, STD)
        post_t = TF.normalize(TF.to_tensor(post_img), MEAN, STD)

        loc_mask = torch.from_numpy(np.array(pre_mask)).float() / 255.0   # (H, W) in [0,1]
        dmg_mask = torch.from_numpy(np.array(post_mask)).long()            # (H, W) in {0,1,2,3,4}

        return pre_t, post_t, loc_mask.unsqueeze(0), dmg_mask


# expB running logs:

Device: cuda
Train samples: 10713  |  Val samples: 1737
Computing damage class weights...
Damage class weights: [0.0, 0.12325441092252731, 1.095670223236084, 1.0693625211715698, 1.7117130756378174]
Loaded Phase 1 weights | missing: 92 | unexpected: 0
/home/vvanera/Desktop/ATML/train_expB.py:145: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 01/40 Train:   0%|                                                 | 0/1340 [00:00<?, ?it/s]/home/vvanera/Desktop/ATML/train_expB.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 01 | train loc 0.0280 dmg 0.0125 acc 0.185 | val loc 0.0263 dmg 0.0131 acc 0.277 | ✓        
Epoch 02 | train loc 0.0183 dmg 0.0093 acc 0.266 | val loc 0.0225 dmg 0.0118 acc 0.277 | ✓        
/home/vvanera/Desktop/ATML/venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)
Epoch 03 | train loc 0.0170 dmg 0.0083 acc 0.281 | val loc 0.0221 dmg 0.0124 acc 0.359 | ✓
Epoch 04 | train loc 0.0165 dmg 0.0077 acc 0.289 | val loc 0.0222 dmg 0.0111 acc 0.329 | ✗        
Epoch 05 | train loc 0.0161 dmg 0.0070 acc 0.350 | val loc 0.0220 dmg 0.0092 acc 0.439 | ✓        
Epoch 06 | train loc 0.0159 dmg 0.0066 acc 0.347 | val loc 0.0218 dmg 0.0094 acc 0.457 | ✓        
Epoch 07 | train loc 0.0157 dmg 0.0062 acc 0.346 | val loc 0.0218 dmg 0.0102 acc 0.383 | ✗        
Epoch 08 | train loc 0.0155 dmg 0.0059 acc 0.384 | val loc 0.0217 dmg 0.0098 acc 0.416 | ✗        
Epoch 09 | train loc 0.0154 dmg 0.0056 acc 0.382 | val loc 0.0219 dmg 0.0087 acc 0.393 | ✗        
Epoch 10 | train loc 0.0152 dmg 0.0052 acc 0.397 | val loc 0.0217 dmg 0.0092 acc 0.427 | ✗        
Epoch 11 | train loc 0.0151 dmg 0.0051 acc 0.408 | val loc 0.0222 dmg 0.0092 acc 0.299 | ✗        
Epoch 12 | train loc 0.0150 dmg 0.0047 acc 0.409 | val loc 0.0217 dmg 0.0085 acc 0.466 | ✓        
Epoch 13 | train loc 0.0148 dmg 0.0044 acc 0.421 | val loc 0.0218 dmg 0.0090 acc 0.443 | ✗        
Epoch 14 | train loc 0.0147 dmg 0.0043 acc 0.422 | val loc 0.0222 dmg 0.0105 acc 0.369 | ✗        
Epoch 15 | train loc 0.0145 dmg 0.0041 acc 0.437 | val loc 0.0228 dmg 0.0094 acc 0.334 | ✗        
Epoch 16 | train loc 0.0144 dmg 0.0039 acc 0.422 | val loc 0.0218 dmg 0.0090 acc 0.465 | ✗        
Epoch 17 | train loc 0.0143 dmg 0.0038 acc 0.465 | val loc 0.0222 dmg 0.0092 acc 0.474 | ✓        
Epoch 18 | train loc 0.0142 dmg 0.0035 acc 0.478 | val loc 0.0225 dmg 0.0095 acc 0.416 | ✗        
Epoch 19 | train loc 0.0140 dmg 0.0033 acc 0.499 | val loc 0.0225 dmg 0.0107 acc 0.453 | ✗        
Epoch 20 | train loc 0.0139 dmg 0.0032 acc 0.506 | val loc 0.0222 dmg 0.0101 acc 0.466 | ✗        
Epoch 21 | train loc 0.0138 dmg 0.0030 acc 0.525 | val loc 0.0224 dmg 0.0101 acc 0.449 | ✗        
Epoch 22 | train loc 0.0137 dmg 0.0029 acc 0.523 | val loc 0.0223 dmg 0.0100 acc 0.544 | ✓        
Epoch 23 | train loc 0.0135 dmg 0.0027 acc 0.553 | val loc 0.0224 dmg 0.0112 acc 0.546 | ✓        
Epoch 24 | train loc 0.0134 dmg 0.0026 acc 0.559 | val loc 0.0235 dmg 0.0113 acc 0.552 | ✓        
Epoch 25 | train loc 0.0133 dmg 0.0024 acc 0.564 | val loc 0.0234 dmg 0.0118 acc 0.636 | ✓        
Epoch 26 | train loc 0.0132 dmg 0.0023 acc 0.570 | val loc 0.0238 dmg 0.0115 acc 0.572 | ✗        
Epoch 27 | train loc 0.0130 dmg 0.0022 acc 0.576 | val loc 0.0233 dmg 0.0116 acc 0.511 | ✗        
Epoch 28 | train loc 0.0130 dmg 0.0020 acc 0.600 | val loc 0.0235 dmg 0.0128 acc 0.604 | ✗        
Epoch 29 | train loc 0.0129 dmg 0.0019 acc 0.591 | val loc 0.0246 dmg 0.0123 acc 0.605 | ✗        
Epoch 30 | train loc 0.0128 dmg 0.0018 acc 0.610 | val loc 0.0240 dmg 0.0141 acc 0.655 | ✓        
Epoch 31 | train loc 0.0127 dmg 0.0017 acc 0.631 | val loc 0.0234 dmg 0.0132 acc 0.643 | ✗        
Epoch 32 | train loc 0.0126 dmg 0.0017 acc 0.641 | val loc 0.0238 dmg 0.0136 acc 0.578 | ✗        
Epoch 33 | train loc 0.0126 dmg 0.0016 acc 0.636 | val loc 0.0242 dmg 0.0143 acc 0.631 | ✗        
Epoch 34 | train loc 0.0125 dmg 0.0015 acc 0.648 | val loc 0.0241 dmg 0.0142 acc 0.630 | ✗        
Epoch 35 | train loc 0.0125 dmg 0.0015 acc 0.648 | val loc 0.0247 dmg 0.0145 acc 0.601 | ✗        
Epoch 36 | train loc 0.0125 dmg 0.0015 acc 0.646 | val loc 0.0254 dmg 0.0144 acc 0.631 | ✗        
Epoch 37 | train loc 0.0124 dmg 0.0014 acc 0.655 | val loc 0.0248 dmg 0.0148 acc 0.648 | ✗        
Epoch 38 | train loc 0.0124 dmg 0.0014 acc 0.653 | val loc 0.0249 dmg 0.0149 acc 0.630 | ✗        
Epoch 39 | train loc 0.0124 dmg 0.0014 acc 0.655 | val loc 0.0249 dmg 0.0151 acc 0.638 | ✗        
Epoch 40 | train loc 0.0124 dmg 0.0015 acc 0.656 | val loc 0.0255 dmg 0.0150 acc 0.662 | ✓        

Best val acc: 0.6619 — saved to best_expB_model.pth

# expC - simclr logs:

python3 train_simclr.py 
Device: cuda
SimCLR pre-training tiles: 10713
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/vvanera/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 212MB/s]
/home/vvanera/Desktop/ATML/train_simclr.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
Epoch 001/50:   0%|                                                                                             | 0/167 [00:00<?, ?it/s]/home/vvanera/Desktop/ATML/train_simclr.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 001 | loss 1.1586 | lr 9.99e-04 | ✓                                                                                               
Epoch 002/50:  89%|██████████████████████████████████████████████████████████████        | 148/167 [10:45<00:51,  2.69s/it, loss=0.5477]Traceback (most recent call last):                                                                                                      
  File "/home/vvanera/Desktop/ATML/train_simclr.py", line 102, in <module>
    vals, counts = nbr[i].unique(return_counts=True)
^^^^^^
  File "/home/vvanera/Desktop/ATML/train_simclr.py", line 60, in main
    return torch.cat(features, dim=0), torch.tensor(labels)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vvanera/Desktop/ATML/venv/lib/python3.12/site-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
  File "/home/vvanera/Desktop/ATML/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 701, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/home/vvanera/Desktop/ATML/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1448, in _next_data
    idx, data = self._get_data()
                ^^^^^^^^^^^^^^^^
  File "/home/vvanera/Desktop/ATML/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1402, in _get_data
    success, data = self._try_get_data()
                    ^^^^^^^^^^^^^^^^^^^^
  File "/home/vvanera/Desktop/ATML/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1243, in _try_get_data
    data = self._data_queue.get(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/queue.py", line 180, in get
    self.not_empty.wait(remaining)
  File "/usr/lib/python3.12/threading.py", line 359, in wait
    gotit = waiter.acquire(True, timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

(venv) vvanera@vclvm176-102:~/Desktop/ATML$ python3 train_simclr.py 
Device: cuda
Train tiles: 10713  |  Val tiles: 1737
Epoch 001 | loss 1.0513 | knn_acc   —    | lr 9.99e-04 |                                                                                
Epoch 002 | loss 0.5298 | knn_acc   —    | lr 9.96e-04 |                                                                                
Epoch 003 | loss 0.3692 | knn_acc   —    | lr 9.91e-04 |                                                                                
Epoch 004 | loss 0.3392 | knn_acc   —    | lr 9.84e-04 |                                                                                
Epoch 005 | loss 0.2738 | knn_acc 0.7028 | lr 9.76e-04 | ✓                                                                              
Epoch 006 | loss 0.2439 | knn_acc   —    | lr 9.65e-04 |                                                                                
Epoch 007 | loss 0.2159 | knn_acc   —    | lr 9.52e-04 |                                                                                
Epoch 008/50:   0%|                                                                                             | 0/167 [00:00<?, ?it/s]Epoch 008 | loss 0.1848 | knn_acc   —    | lr 9.38e-04 |                                                                                
Epoch 009 | loss 0.1768 | knn_acc   —    | lr 9.22e-04 |                                                                                
Epoch 010 | loss 0.1709 | knn_acc 0.6834 | lr 9.05e-04 |                                                                                
Epoch 011 | loss 0.1525 | knn_acc   —    | lr 8.85e-04 |                                                                                
Epoch 012/50:  22%|███████████████▎                                                       | 36/167 [02:45<05:35,  2.56s/it, loss=0.1456                                      Epoch 012/50:  22%|███████████████▋                                                       | 37/167 [02:45<12:13,  5.64s/it, loss=0.1456                                      Epoch 012/50:  22%|███████████████▋                                                       | 37/167 [02:50<12:13,  5.64s/it, loss=0.1440                                      Epoch 012/50:  23%|████████████████▏                                                      | 38/167 [02:50<11:36,  5.40s/it, loss=0.1440                 Epoch 012/50:  23%|████████████████▏                                                      | 38/167 [02:50<11:36,  5.40s/it, loss=0.1434                                      Epoch 012/50:  23%|████████████████▌                                                      | 39/167 [02:50<08:27,  3.96s/it, loss=0.1434                                      Epoch 012/50:  23%|████████████████▌                                                      | 39/167 [02:51<08:27,  3.96s/it, loss=0.1489                                      Epoch 012/50:  24%|█████████████████                                                      | 40/167 [02:51<06:14,  2.95s/it, loss=0.1489                                  Epoch 012/50:  24%|█████████████████                                                      | 40/167 [03:00<06:14,  2.95s/it, loss=0.1482                                      Epoch 012/50:  25%|█████████████████▍                                                     | 41/167 [03:00<10:09,  4.84s/it, loss=0.1482                                      Epoch 012/50:  25%|█████████████████▍                                                     | 41/167 [03:09<10:09,  4.84s/it, loss=0.1486             Epoch 012/50:  25%|█████████████████▊                                                     | 42/167 [03:09<12:35,  6.05s/it, loss=0.1486                                      Epoch 012/50:  25%|█████████████████▊                                                     | 42/167 [03:10<12:35,  6.05s/it, loss=0.1488                                      Epoch 012/50:  26%|██████████████████▎                                                    | 43/167 [03:10<09:07,  4.41s/it, loss=0.1488                                      Epoch 012/50:  26%|██████████████████▎                                                    | 43/167 [03:10<09:07,  4.41s/it, loss=0.1487                              Epoch 012/50:  26%|██████████████████▋                                                    | 44/167 [03:10<06:41,  3.27s/it, loss=0.1487                                      Epoch 012 | loss 0.1525 | knn_acc   —    | lr 8.64e-04 |                                                                                
Epoch 013 | loss 0.1282 | knn_acc   —    | lr 8.42e-04 |                                                                                                                     
Epoch 014 | loss 0.1045 | knn_acc   —    | lr 8.19e-04 |                                                                                                                     
Epoch 015 | loss 0.1110 | knn_acc 0.6867 | lr 7.94e-04 |                                                                                                                     
Epoch 016 | loss 0.0957 | knn_acc   —    | lr 7.68e-04 |                                                                                                                     
Epoch 017 | loss 0.1033 | knn_acc   —    | lr 7.41e-04 |                                                                                                                     
Epoch 018 | loss 0.0905 | knn_acc   —    | lr 7.13e-04 |                                                                                                                     
Epoch 019 | loss 0.0759 | knn_acc   —    | lr 6.84e-04 |                                                                                                                     
Epoch 020 | loss 0.0813 | knn_acc 0.6780 | lr 6.55e-04 |                                                                                                                     
Epoch 021 | loss 0.0647 | knn_acc   —    | lr 6.24e-04 |                                                                                                                     
Epoch 022 | loss 0.0602 | knn_acc   —    | lr 5.94e-04 |                                                                                                                     
Epoch 023 | loss 0.0631 | knn_acc   —    | lr 5.63e-04 |                                                                                                                     
Epoch 024 | loss 0.0551 | knn_acc   —    | lr 5.31e-04 |                                                                                                                     
Epoch 025 | loss 0.0548 | knn_acc 0.6990 | lr 5.00e-04 |                                                                                                                     
Epoch 026 | loss 0.0470 | knn_acc   —    | lr 4.69e-04 |                                                                                                                     
Epoch 027 | loss 0.0424 | knn_acc   —    | lr 4.37e-04 |                                                                                                                     
Epoch 028 | loss 0.0483 | knn_acc   —    | lr 4.06e-04 |                                                                                                                     
Epoch 029 | loss 0.0359 | knn_acc   —    | lr 3.76e-04 |                                                                                                                     
Epoch 030 | loss 0.0323 | knn_acc 0.6579 | lr 3.45e-04 |                                                                                                                     
Epoch 031 | loss 0.0320 | knn_acc   —    | lr 3.16e-04 |                                                                                                                     
Epoch 032 | loss 0.0260 | knn_acc   —    | lr 2.87e-04 |                                                                                                                     
Epoch 033 | loss 0.0258 | knn_acc   —    | lr 2.59e-04 |                                                                                                                     
Epoch 034 | loss 0.0234 | knn_acc   —    | lr 2.32e-04 |                                                                                                                     
Epoch 035 | loss 0.0219 | knn_acc 0.6539 | lr 2.06e-04 |

# train_expC.py

In [ ]:
import os
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset_expB import XBDJointDataset
from model_expC import JointDamageNet

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR         = "xbd"
TRAIN_SPLITS     = ["tier1", "tier3"]
VAL_SPLITS       = ["hold"]
BATCH_SIZE       = 8
NUM_WORKERS      = 4
LR               = 1e-4
ENCODER_LR_MULT  = 0.3        # encoder gets LR * 0.3 (preserve Phase 1 features)
EPOCHS           = 40
WARMUP_EPOCHS    = 3
FOCAL_GAMMA      = 2.0
LOC_LOSS_WEIGHT  = 1.0
DMG_LOSS_WEIGHT  = 1.0
CONTRAST_WEIGHT  = 0.1        # auxiliary contrastive loss weight
CONTRAST_TEMP    = 0.1        # SupCon temperature
CONTRAST_SAMPLES = 512        # max building pixels sampled per batch for SupCon
DROPOUT          = 0.3
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH        = "best_expC_model.pth"
PHASE1_CKPT      = "best_model.pth"
# ──────────────────────────────────────────────────────────────────────────────


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=0):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight,
                               ignore_index=self.ignore_index, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt  = torch.exp(-bce)
        return (((1 - pt) ** self.gamma) * bce).mean()


def pixel_supcon_loss(embeddings, labels, temperature=0.1, max_samples=512):
    """
    Pixel-level Supervised Contrastive Loss.

    Samples building pixels from the batch, projects them to an embedding space,
    and pushes same-damage-class pixels together while pulling different classes apart.

    Args:
        embeddings: (B, D, H, W) — projected features at stride 16
        labels:     (B, H, W)    — damage class labels downsampled to match
                                    (0=background/ignored, 1–4=damage classes)
        temperature: scaling factor for cosine similarity
        max_samples: cap on sampled pixels to limit memory
    """
    B, D, H, W = embeddings.shape

    # Cast to float32 and replace any nan/inf that may come from float16 BN
    # in contrast_proj during early training with random initialization.
    embeddings = torch.nan_to_num(embeddings.float(), nan=0.0, posinf=0.0, neginf=0.0)

    feats = embeddings.permute(0, 2, 3, 1).reshape(-1, D)   # (N, D)
    labs  = labels.reshape(-1)                               # (N,)

    # Only building pixels (label > 0)
    building_mask = labs > 0
    feats = feats[building_mask]
    labs  = labs[building_mask]
    N     = feats.size(0)

    if N < 2:
        return (embeddings * 0).sum()  # differentiable zero

    # Subsample to keep memory bounded
    if N > max_samples:
        idx   = torch.randperm(N, device=feats.device)[:max_samples]
        feats = feats[idx]
        labs  = labs[idx]
        N     = max_samples

    feats = F.normalize(feats, dim=1)                         # L2-normalise
    sim   = feats @ feats.T / temperature                     # (N, N)

    # Masks
    self_mask = torch.eye(N, dtype=torch.bool, device=feats.device)
    pos_mask  = (labs.unsqueeze(0) == labs.unsqueeze(1)) & ~self_mask

    pos_counts = pos_mask.sum(dim=1)
    valid      = pos_counts > 0
    if valid.sum() == 0:
        return (embeddings * 0).sum()

    sim      = sim.masked_fill(self_mask, float('-inf'))
    log_prob = F.log_softmax(sim, dim=1)  # diagonal is -inf here

    # Zero out diagonal explicitly before multiplying to avoid -inf * 0 = nan
    log_prob = log_prob.masked_fill(self_mask, 0.0)

    mean_log_prob = (log_prob * pos_mask.float()).sum(dim=1) / pos_counts.clamp(min=1)
    return -mean_log_prob[valid].mean()


def compute_dmg_weights(dataset):
    counts = collections.Counter()
    for _, _, _, post_mask_path, _, _ in dataset.tiles:
        from PIL import Image
        import numpy as np
        arr = np.array(Image.open(post_mask_path)).flatten()
        for v in range(1, 5):
            counts[v] += (arr == v).sum()
    total = sum(counts.values())
    weights = torch.tensor(
        [total / (4 * max(counts[c], 1)) for c in range(1, 5)], dtype=torch.float32
    )
    weights = weights / weights.sum() * 4
    return torch.cat([torch.zeros(1), weights])


def run_epoch(model, loader, loc_criterion, dmg_criterion, optimizer, scaler,
              train, epoch, num_epochs):
    model.train(train)
    total_loc, total_dmg, total_con = 0.0, 0.0, 0.0
    total_dmg_correct, total_dmg_pixels, n = 0, 0, 0
    phase = "Train" if train else "Val"
    pbar  = tqdm(loader, desc=f"Epoch {epoch:02d}/{num_epochs} {phase}", leave=False)

    with torch.set_grad_enabled(train):
        for pre, post, loc_mask, dmg_mask in pbar:
            pre      = pre.to(DEVICE)
            post     = post.to(DEVICE)
            loc_mask = loc_mask.to(DEVICE)
            dmg_mask = dmg_mask.to(DEVICE)

            with autocast("cuda"):
                loc_out, dmg_out, contrast_embed = model(pre, post)

                loc_loss = loc_criterion(loc_out, loc_mask)
                dmg_loss = dmg_criterion(dmg_out, dmg_mask)

                # Upsample contrast_embed to full resolution so the loss sees
                # all building pixels rather than the tiny stride-16 map.
                contrast_embed_full = F.interpolate(
                    contrast_embed,
                    size=dmg_mask.shape[1:],
                    mode="bilinear", align_corners=False,
                )

                con_loss = pixel_supcon_loss(
                    contrast_embed_full, dmg_mask,
                    temperature=CONTRAST_TEMP,
                    max_samples=CONTRAST_SAMPLES,
                )

                con_valid = torch.isfinite(con_loss)
                loss = (LOC_LOSS_WEIGHT * loc_loss
                        + DMG_LOSS_WEIGHT * dmg_loss
                        + (CONTRAST_WEIGHT * con_loss if con_valid else 0.0))

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            bs   = pre.size(0)
            mask = dmg_mask > 0
            total_loc         += loc_loss.item() * bs
            total_dmg         += dmg_loss.item() * bs
            if con_valid:
                total_con     += con_loss.item() * bs
            total_dmg_correct += (dmg_out.argmax(1)[mask] == dmg_mask[mask]).sum().item()
            total_dmg_pixels  += mask.sum().item()
            n += bs

            pbar.set_postfix(
                loc=f"{total_loc/n:.4f}",
                dmg=f"{total_dmg/n:.4f}",
                con=f"{total_con/n:.4f}",
                acc=f"{total_dmg_correct/max(total_dmg_pixels,1):.3f}",
            )

    avg_acc = total_dmg_correct / max(total_dmg_pixels, 1)
    return total_loc / n, total_dmg / n, total_con / n, avg_acc


def main():
    print(f"Device: {DEVICE}")

    train_ds = XBDJointDataset(TRAIN_SPLITS, BASE_DIR, augment=True)
    val_ds   = XBDJointDataset(VAL_SPLITS,   BASE_DIR, augment=False)
    print(f"Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}")

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print("Computing damage class weights...")
    dmg_weights = compute_dmg_weights(train_ds).to(DEVICE)
    print("Damage class weights:", dmg_weights.tolist())

    model = JointDamageNet(loc_classes=1, dmg_classes=5, dropout=DROPOUT).to(DEVICE)
    model.load_phase1_weights(PHASE1_CKPT, DEVICE)

    loc_criterion = BinaryFocalLoss(gamma=FOCAL_GAMMA)
    dmg_criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=dmg_weights, ignore_index=0)

    # Encoder: lower LR to preserve Phase 1 features
    # Decoder heads + contrast proj: full LR
    optimizer = torch.optim.AdamW([
        {"params": model.low_level.parameters(),     "lr": LR * ENCODER_LR_MULT},
        {"params": model.high_level.parameters(),    "lr": LR * ENCODER_LR_MULT},
        {"params": model.loc_head.parameters(),      "lr": LR},
        {"params": model.dmg_head.parameters(),      "lr": LR},
        {"params": model.contrast_proj.parameters(), "lr": LR},
    ], weight_decay=1e-4)

    scaler = GradScaler("cuda")

    best_val_acc = 0.0
    start_epoch  = 1

    if os.path.exists(SAVE_PATH):
        print(f"Resuming from checkpoint: {SAVE_PATH}")
        ckpt         = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_val_acc = ckpt.get("val_acc", 0.0)
        start_epoch  = ckpt["epoch"] + 1
        print(f"Resumed at epoch {start_epoch}, best val acc: {best_val_acc:.4f}")

    warmup    = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                                                   total_iters=WARMUP_EPOCHS)
    cosine    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine],
                                                       milestones=[WARMUP_EPOCHS])

    if os.path.exists(SAVE_PATH):
        ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        if "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loc, tr_dmg, tr_con, tr_acc = run_epoch(
            model, train_loader, loc_criterion, dmg_criterion,
            optimizer, scaler, train=True, epoch=epoch, num_epochs=EPOCHS)
        vl_loc, vl_dmg, vl_con, vl_acc = run_epoch(
            model, val_loader, loc_criterion, dmg_criterion,
            optimizer, scaler, train=False, epoch=epoch, num_epochs=EPOCHS)
        scheduler.step()

        improved = vl_acc > best_val_acc
        if improved:
            best_val_acc = vl_acc
            torch.save({
                "epoch":                epoch,
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict":    scaler.state_dict(),
                "val_acc":              vl_acc,
            }, SAVE_PATH)

        status = "✓" if improved else " "
        print(f"Epoch {epoch:02d} | "
              f"train loc {tr_loc:.4f} dmg {tr_dmg:.4f} con {tr_con:.4f} acc {tr_acc:.3f} | "
              f"val loc {vl_loc:.4f} dmg {vl_dmg:.4f} con {vl_con:.4f} acc {vl_acc:.3f} | {status}")

    print(f"\nBest val acc: {best_val_acc:.4f} — saved to {SAVE_PATH}")


if __name__ == "__main__":
    main()


# model_expC.py

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels=256):
        super().__init__()
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=True),
            nn.GroupNorm(32, out_channels), nn.ReLU()
        )
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False),
                          nn.BatchNorm2d(out_channels), nn.ReLU()),
            *[nn.Sequential(nn.Conv2d(in_channels, out_channels, 3,
                                      padding=r, dilation=r, bias=False),
                            nn.BatchNorm2d(out_channels), nn.ReLU())
              for r in (6, 12, 18)],
        ])
        self.proj = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Dropout(0.5)
        )

    def forward(self, x):
        size = x.shape[2:]
        pool = F.interpolate(self.global_pool(x), size, mode="bilinear", align_corners=False)
        feats = [pool] + [b(x) for b in self.branches]
        return self.proj(torch.cat(feats, dim=1))


class DecoderHead(nn.Module):
    def __init__(self, num_classes, high_in, low_in, dropout=0.3):
        super().__init__()
        self.aspp     = ASPP(high_in)
        self.low_proj = nn.Sequential(
            nn.Conv2d(low_in, 48, 1, bias=False),
            nn.BatchNorm2d(48), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(256, num_classes, 1)
        )

    def forward(self, low, high):
        aspp = self.aspp(high)
        aspp = F.interpolate(aspp, low.shape[2:], mode="bilinear", align_corners=False)
        x    = torch.cat([aspp, self.low_proj(low)], dim=1)
        x    = self.decoder(x)
        return F.interpolate(x, scale_factor=4, mode="bilinear", align_corners=False)


class JointDamageNet(nn.Module):
    def __init__(self, loc_classes=1, dmg_classes=5, dropout=0.3):
        super().__init__()
        backbone = resnet50(weights=None)

        self.low_level = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1
        )  # -> (B, 256, H/4, W/4)

        layer3 = backbone.layer3
        for m in layer3.modules():
            if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
                m.stride = (1, 1)
            if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3):
                m.dilation = (2, 2)
                m.padding  = (2, 2)

        self.high_level = nn.Sequential(backbone.layer2, layer3)
        # -> (B, 1024, H/16, W/16)

        # Localization head: cat(pre, post) — same as expB
        self.loc_head = DecoderHead(
            num_classes=loc_classes,
            high_in=1024 * 2, low_in=256 * 2,
            dropout=dropout,
        )

        # Damage head: cat(pre, post, post-pre) — includes explicit change signal
        self.dmg_head = DecoderHead(
            num_classes=dmg_classes,
            high_in=1024 * 3, low_in=256 * 3,
            dropout=dropout,
        )

        # Projection head for pixel-level contrastive loss on difference features
        self.contrast_proj = nn.Sequential(
            nn.Conv2d(1024, 256, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 128, 1, bias=False),
        )

    def _encode(self, x):
        low  = self.low_level(x)
        high = self.high_level(low)
        return low, high

    def forward(self, pre, post):
        pre_low,  pre_high  = self._encode(pre)
        post_low, post_high = self._encode(post)

        # ── Localization: cat(pre, post) — same as expB ──────────────────────
        loc_low  = torch.cat([pre_low,  post_low],  dim=1)  # (B, 512,  H/4,  W/4)
        loc_high = torch.cat([pre_high, post_high], dim=1)   # (B, 2048, H/16, W/16)
        loc_out  = self.loc_head(loc_low, loc_high)

        # ── Damage: cat(pre, post, post-pre) — explicit change signal ────────
        low_diff  = post_low  - pre_low                      # (B, 256,  H/4,  W/4)
        high_diff = post_high - pre_high                     # (B, 1024, H/16, W/16)

        dmg_low  = torch.cat([pre_low,  post_low,  low_diff],  dim=1)  # (B, 768,  H/4,  W/4)
        dmg_high = torch.cat([pre_high, post_high, high_diff], dim=1)   # (B, 3072, H/16, W/16)
        dmg_out  = self.dmg_head(dmg_low, dmg_high)

        # ── Contrastive embeddings from change features ──────────────────────
        contrast_embed = self.contrast_proj(high_diff)       # (B, 128, H/16, W/16)

        return loc_out, dmg_out, contrast_embed

    def load_phase1_weights(self, ckpt_path, device):
        ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
        state = ckpt["model_state_dict"]

        new_state = {k: v for k, v in state.items()
                     if k.startswith("low_level.") or k.startswith("high_level.")}

        missing, unexpected = self.load_state_dict(new_state, strict=False)
        print(f"Loaded Phase 1 weights | missing: {len(missing)} | unexpected: {len(unexpected)}")


# dataset_simclr.py

In [ ]:
import os
import glob
import random

import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

MEAN      = [0.485, 0.456, 0.406]
STD       = [0.229, 0.224, 0.225]
TILE_SIZE = 512


def _color_aug(img):
    """
    Independent per-view colour augmentation.
    Kept mild for aerial imagery where colour carries semantic meaning.
    """
    if random.random() < 0.8:
        img = TF.adjust_contrast(img,   random.uniform(0.6, 1.4))
        img = TF.adjust_brightness(img, random.uniform(0.6, 1.4))
        img = TF.adjust_saturation(img, random.uniform(0.6, 1.4))
        img = TF.adjust_hue(img,        random.uniform(-0.1, 0.1))
    if random.random() < 0.2:
        img = TF.rgb_to_grayscale(img, num_output_channels=3)
    if random.random() < 0.3:
        img = TF.gaussian_blur(img, kernel_size=23,
                               sigma=random.uniform(0.1, 2.0))
    return img


class XBDSimCLRDataset(Dataset):
    """
    Positive pair = (pre tile, post tile) from the same scene location.

    Spatial augmentations (flip, rotate) are shared between pre and post
    so they remain spatially aligned. Colour augmentations are applied
    independently to each view so the encoder cannot trivially match them
    by low-level colour statistics alone.

    The contrastive objective forces the encoder to learn that pre and post
    images of the same location represent the same underlying scene structure,
    regardless of disaster-induced appearance changes.
    """
    def __init__(self, splits, base_dir="xbd", min_building_ratio=0.01):
        self.tiles = []  # (pre_img_path, post_img_path, row, col)

        for split in splits:
            img_dir  = os.path.join(base_dir, split, "images")
            mask_dir = os.path.join(base_dir, split, "masks")
            if not os.path.isdir(img_dir):
                continue

            for pre_img_path in glob.glob(os.path.join(img_dir, "*_pre_disaster.png")):
                stem          = os.path.basename(pre_img_path).replace("_pre_disaster.png", "")
                post_img_path = os.path.join(img_dir,  f"{stem}_post_disaster.png")
                pre_mask_path = os.path.join(mask_dir, f"{stem}_pre_disaster.png")

                if not all(os.path.exists(p) for p in [post_img_path, pre_mask_path]):
                    continue

                pre_mask = np.array(Image.open(pre_mask_path))
                h, w     = pre_mask.shape
                for r in range(0, h, TILE_SIZE):
                    for c in range(0, w, TILE_SIZE):
                        tile = pre_mask[r:r + TILE_SIZE, c:c + TILE_SIZE]
                        if tile.shape != (TILE_SIZE, TILE_SIZE):
                            continue
                        ratio = (tile > 0).sum() / (TILE_SIZE * TILE_SIZE)
                        if ratio >= min_building_ratio:
                            self.tiles.append((pre_img_path, post_img_path, r, c))

    def __len__(self):
        return len(self.tiles)

    def get_tile_label(self, idx):
        """
        Returns the dominant non-zero damage class for tile idx.
        Derived from the post-disaster mask (0=bg, 1-4=damage severity).
        Used by the KNN validation probe in train_simclr.py.
        """
        _, post_img_path, r, c = self.tiles[idx]
        post_mask_path = post_img_path.replace("/images/", "/masks/")
        if not os.path.exists(post_mask_path):
            return 0
        arr  = np.array(Image.open(post_mask_path))
        tile = arr[r:r + TILE_SIZE, c:c + TILE_SIZE]
        building = tile[tile > 0]
        if len(building) == 0:
            return 0
        vals, counts = np.unique(building, return_counts=True)
        return int(vals[counts.argmax()])

    def __getitem__(self, idx):
        pre_path, post_path, r, c = self.tiles[idx]

        def load_crop(path):
            arr = np.array(Image.open(path).convert("RGB"))
            return Image.fromarray(arr[r:r + TILE_SIZE, c:c + TILE_SIZE])

        pre_img  = load_crop(pre_path)
        post_img = load_crop(post_path)

        # ── Shared spatial augmentations (pre and post stay aligned) ──────────
        if random.random() > 0.5:
            pre_img  = TF.hflip(pre_img)
            post_img = TF.hflip(post_img)
        if random.random() > 0.5:
            pre_img  = TF.vflip(pre_img)
            post_img = TF.vflip(post_img)
        k = random.randint(0, 3)
        if k:
            pre_img  = TF.rotate(pre_img,  90 * k)
            post_img = TF.rotate(post_img, 90 * k)

        # ── Independent colour augmentations (each view looks different) ──────
        pre_img  = _color_aug(pre_img)
        post_img = _color_aug(post_img)

        to_t = lambda img: TF.normalize(TF.to_tensor(img), MEAN, STD)
        return to_t(pre_img), to_t(post_img)


# model_simclr.py

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights


class SimCLREncoder(nn.Module):
    """
    ResNet50 backbone (ImageNet init) + MLP projection head for SimCLR.
    Backbone components are kept as named attributes so weights can be
    transferred into JointDamageNet via load_simclr_weights().
    """
    def __init__(self, proj_dim=128, proj_hidden=2048):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        # Keep as named attrs — state dict keys will be "conv1.*", "layer1.*", etc.
        self.conv1   = backbone.conv1
        self.bn1     = backbone.bn1
        self.relu    = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.layer4  = backbone.layer4
        self.avgpool = backbone.avgpool  # AdaptiveAvgPool2d(1) → (B, 2048, 1, 1)

        # Projection head: 2048 → proj_hidden → proj_dim
        # BN after each linear, no affine on the final BN (SimCLR v2 convention)
        self.projector = nn.Sequential(
            nn.Linear(2048, proj_hidden, bias=False),
            nn.BatchNorm1d(proj_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(proj_hidden, proj_dim, bias=False),
            nn.BatchNorm1d(proj_dim, affine=False),
        )

    def encode(self, x):
        """Backbone only — used at transfer time, not during pre-training."""
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)  # (B, 2048)

    def forward(self, x):
        h = self.encode(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)  # L2-normalised


def nt_xent_loss(z1, z2, temperature=0.07):
    """
    NT-Xent (Normalised Temperature-scaled Cross-Entropy) loss.
    z1, z2: (N, D) L2-normalised embeddings of N positive pairs.
    Each z1[i] has z2[i] as its positive; all other 2N-2 embeddings are negatives.
    """
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)          # (2N, D)
    sim = torch.mm(z, z.T) / temperature     # (2N, 2N) cosine similarities

    # Mask out self-similarities (diagonal)
    mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim  = sim.masked_fill(mask, float('-inf'))

    # For z1[i] the positive is z2[i] = z[N+i]; for z2[i] the positive is z1[i] = z[i]
    labels = torch.cat([
        torch.arange(N, 2 * N, device=z.device),
        torch.arange(0,     N, device=z.device),
    ])

    return F.cross_entropy(sim, labels)


# train_simclr.py

In [ ]:
import os

import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset_simclr import XBDSimCLRDataset, MEAN, STD, TILE_SIZE
from model_simclr import SimCLREncoder, nt_xent_loss
import torchvision.transforms.functional as TF

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR        = "xbd"
TRAIN_SPLITS    = ["tier1", "tier3"]
VAL_SPLITS      = ["hold"]
BATCH_SIZE      = 64            # larger = more negatives = better; reduce if OOM
NUM_WORKERS     = 4
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
EPOCHS          = 50
TEMPERATURE     = 0.07
PROJ_DIM        = 128
KNN_K           = 5             # neighbours for KNN probe
KNN_TRAIN_N     = 1000          # training samples for KNN feature bank
KNN_VAL_N       = 500           # validation samples to evaluate
KNN_EVAL_FREQ   = 5             # run KNN probe every N epochs
SAVE_PATH       = "best_simclr.pth"
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ──────────────────────────────────────────────────────────────────────────────


def extract_features(model, dataset, indices, device, batch_size=64):
    """
    Extract 2048-d global encoder features from post-disaster tiles (no augmentation).
    Returns (features, labels) — labels are the dominant damage class per tile.
    Post images are used because they directly show the damage state.
    """
    features, labels = [], []

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        imgs, batch_labels = [], []

        for idx in batch_idx:
            _, post_path, r, c = dataset.tiles[idx]
            arr  = np.array(Image.open(post_path).convert("RGB"))
            tile = Image.fromarray(arr[r:r + TILE_SIZE, c:c + TILE_SIZE])
            imgs.append(TF.normalize(TF.to_tensor(tile), MEAN, STD))
            batch_labels.append(dataset.get_tile_label(idx))

        batch_tensor = torch.stack(imgs).to(device)
        with torch.no_grad(), autocast("cuda"):
            feats = model.encode(batch_tensor)           # (B, 2048)
        features.append(F.normalize(feats, dim=1).cpu())
        labels.extend(batch_labels)

    return torch.cat(features, dim=0), torch.tensor(labels)


@torch.no_grad()
def evaluate_knn(model, train_ds, val_ds, device,
                 k=5, n_train=1000, n_val=500, batch_size=64):
    """
    KNN probe accuracy using frozen encoder features.

    - Builds a feature bank from n_train random training tiles.
    - Classifies n_val validation tiles by majority vote among k nearest neighbours.
    - Both sets filtered to tiles with a non-zero damage label.
    - Features extracted from post-disaster images (show damage state).
    """
    model.eval()

    train_idx = torch.randperm(len(train_ds))[:n_train].tolist()
    val_idx   = torch.randperm(len(val_ds))[:n_val].tolist()

    train_feats, train_labels = extract_features(model, train_ds, train_idx, device, batch_size)
    val_feats,   val_labels   = extract_features(model, val_ds,   val_idx,   device, batch_size)

    # Keep only tiles with a valid damage label
    tr_mask = train_labels > 0
    vl_mask = val_labels   > 0

    if tr_mask.sum() < k or vl_mask.sum() == 0:
        return 0.0

    tf = train_feats[tr_mask]
    tl = train_labels[tr_mask]
    vf = val_feats[vl_mask]
    vl = val_labels[vl_mask]

    # Cosine similarity (features are already L2-normalised)
    sim     = vf @ tf.T                    # (n_val, n_train)
    _, topk = sim.topk(k, dim=1)           # (n_val, k)
    nbr     = tl[topk]                     # (n_val, k) neighbour labels

    # Majority vote
    preds = torch.zeros(len(vf), dtype=torch.long)
    for i in range(len(vf)):
        vals, counts = nbr[i].unique(return_counts=True)
        preds[i] = vals[counts.argmax()]

    return (preds == vl).float().mean().item()


def main():
    print(f"Device: {DEVICE}")

    train_ds = XBDSimCLRDataset(TRAIN_SPLITS, BASE_DIR)
    val_ds   = XBDSimCLRDataset(VAL_SPLITS,   BASE_DIR)
    print(f"Train tiles: {len(train_ds)}  |  Val tiles: {len(val_ds)}")

    # drop_last=True keeps every batch full — NT-Xent needs consistent N
    loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    model     = SimCLREncoder(proj_dim=PROJ_DIM).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = GradScaler("cuda")

    best_val_acc = 0.0
    start_epoch  = 1

    if os.path.exists(SAVE_PATH):
        print(f"Resuming from {SAVE_PATH}")
        ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        best_val_acc = ckpt.get("val_knn_acc", 0.0)
        start_epoch  = ckpt["epoch"] + 1
        print(f"Resumed at epoch {start_epoch}, best val KNN acc: {best_val_acc:.4f}")

    for epoch in range(start_epoch, EPOCHS + 1):
        # ── Training loop ─────────────────────────────────────────────────────
        model.train()
        total_loss, n = 0.0, 0
        pbar = tqdm(loader, desc=f"Epoch {epoch:03d}/{EPOCHS}", leave=False)

        for view1, view2 in pbar:
            view1 = view1.to(DEVICE)
            view2 = view2.to(DEVICE)

            optimizer.zero_grad()
            with autocast("cuda"):
                z1   = model(view1)
                z2   = model(view2)
                loss = nt_xent_loss(z1, z2, TEMPERATURE)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs         = view1.size(0)
            total_loss += loss.item() * bs
            n          += bs
            pbar.set_postfix(loss=f"{total_loss / n:.4f}")

        scheduler.step()
        epoch_loss = total_loss / n

        # ── KNN validation probe (every KNN_EVAL_FREQ epochs) ─────────────────
        val_knn_acc = best_val_acc  # carry forward if not evaluating this epoch
        if epoch % KNN_EVAL_FREQ == 0 or epoch == EPOCHS:
            val_knn_acc = evaluate_knn(
                model, train_ds, val_ds, DEVICE,
                k=KNN_K, n_train=KNN_TRAIN_N, n_val=KNN_VAL_N,
            )

        improved = val_knn_acc > best_val_acc
        if improved:
            best_val_acc = val_knn_acc
            torch.save({
                "epoch":                epoch,
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict":    scaler.state_dict(),
                "val_knn_acc":          val_knn_acc,
                "train_loss":           epoch_loss,
            }, SAVE_PATH)

        status = "✓" if improved else " "
        knn_str = f"{val_knn_acc:.4f}" if epoch % KNN_EVAL_FREQ == 0 or epoch == EPOCHS else "  —   "
        print(f"Epoch {epoch:03d} | loss {epoch_loss:.4f} "
              f"| knn_acc {knn_str} "
              f"| lr {scheduler.get_last_lr()[0]:.2e} | {status}")

    print(f"\nBest val KNN acc: {best_val_acc:.4f} — saved to {SAVE_PATH}")


if __name__ == "__main__":
    main()
